# Fundamentos de NLP con spaCy

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ohtar10/icesi-nlp/blob/main/Sesion1/1-fundamentos-nlp-con-spacy.ipynb)

Este notebook compacta la base de NLP clásico necesaria para arrancar el curso: documentos, tokens, spans, oraciones, tokenización, entidades y visualización sintáctica con spaCy. La meta es que desde la primera sesión puedas inspeccionar texto, entender cómo lo segmenta un pipeline y reconocer qué información lingüística queda disponible para tareas posteriores.

## Referencias
* [NLP - Natural Language Processing With Python](https://www.udemy.com/course/nlp-natural-language-processing-with-python)
* [Natural Language Processing in Action](https://www.manning.com/books/natural-language-processing-in-action)
* [spaCy Usage Documentation](https://spacy.io/usage)

## Preparación del entorno
Asumiendo que la librería ya se encuentra instalada, dependiendo de la tarea, necesitamos descargar el modelo o las dependencias puntuales antes de empezar.


In [1]:
import warnings

warnings.filterwarnings('ignore')

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [2]:
!test '{IN_COLAB}' = 'True' && wget  https://github.com/Ohtar10/icesi-nlp/raw/refs/heads/main/requirements.txt && pip install -r requirements.txt

In [3]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 43.2 MB/s  0:00:00eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


El cual debemos luego importar:

In [4]:
import spacy

# load the simplified version of the english core language
nlp = spacy.load('en_core_web_sm')

## Creando un documento simple
Este documento será automáticamente interpretado con spacy para el lenguaje seleccionado.

In [5]:
doc = nlp(u'Tesla is looking at buying U.S. startup for $6 million')

Desde aquí, podemos observar los diferentes elementos del documento.

In [6]:
col1 = "Token"
col2 = "POS" # Part of Speech
col3 = "S-dep" # Syntactic dependency

print(f"{col1:{20}}{col2:{20}}{col3:{20}}")
for token in doc:
    print(f"{token.text:{20}}{token.pos_:{20}}{token.dep_}")

Token               POS                 S-dep               
Tesla               PROPN               nsubj
is                  AUX                 aux
looking             VERB                ROOT
at                  ADP                 prep
buying              VERB                pcomp
U.S.                PROPN               dobj
startup             VERB                advcl
for                 ADP                 prep
$                   SYM                 quantmod
6                   NUM                 compound
million             NUM                 pobj


Hemos impreso los tokens (palabras en este caso), la parte del contexto que representan (POS) y la dependencia semantica que dicho token tiene.

En NLP clásico hay una taxonomía especializada para cada elemento del lenguaje. Cada elemento fue producto de estudios diversos y variados con el fin de ofrecer un modelado sistemático del lenguaje. Expertos en lenguaje estuvieron involucrados en la creación de esta taxonomía.

Ahora, librerías como spaCy facilitan el procesamiento de esta taxonomía.

## Un pipeline simple de spaCy

El núcleo de spaCy es el pipeline que no es más que el procesamiento/transformación que toma el texto original y se lo somete a diversos procesos de NLP

In [7]:
nlp.pipeline

[('tok2vec', <spacy.pipeline.tok2vec.Tok2Vec at 0x7dbe96271250>),
 ('tagger', <spacy.pipeline.tagger.Tagger at 0x7dbe96271310>),
 ('parser', <spacy.pipeline.dep_parser.DependencyParser at 0x7dbe9626d150>),
 ('attribute_ruler',
  <spacy.pipeline.attributeruler.AttributeRuler at 0x7dbe9601ae10>),
 ('lemmatizer',
  <spacy.lang.en.lemmatizer.EnglishLemmatizer at 0x7dbe9601bc50>),
 ('ner', <spacy.pipeline.ner.EntityRecognizer at 0x7dbe9626d0e0>)]

Como podemos observar aquí, la instanciación por defecto es un pipeline compuesto por diferentes componentes que deberían ser familiares para nosotros:

* Token 2 Vec: Convertir los tokens en vectores.
* Lemmatizer: Extracción de componentes raíz de las palabras
* NER: Named entity recognition para identificar los sujetos de los documentos.

Un documento es iterable y los items pueden ser accedidos por índice.

In [8]:
n = 0
print(f"The {n}th token in the document is: {doc[n]}")

The 0th token in the document is: Tesla


## Exploremos diferentes elementos transformados

In [9]:
from spacy.tokens.doc import Doc
import pandas as pd

def get_doc_elements(doc: Doc):
    elements = ["text", "lemma", "pos", "tag", "shape", "alpha", "stop"]
    rows = [ [token.text, token.lemma_, token.pos_, token.tag_, token.shape_, token.is_alpha, token.is_stop] 
            for token in  doc]
    return pd.DataFrame(rows, columns=elements)

In [10]:
doc_elements = get_doc_elements(doc)
doc_elements

,text,lemma,pos,tag,shape,alpha,stop
0,Tesla,Tesla,PROPN,NNP,Xxxxx,True,False
1,is,be,AUX,VBZ,xx,True,True
2,looking,look,VERB,VBG,xxxx,True,False
3,at,at,ADP,IN,xx,True,True
4,buying,buy,VERB,VBG,xxxx,True,False
5,U.S.,U.S.,PROPN,NNP,X.X.,False,False
6,startup,startup,VERB,VBD,xxxx,True,False
7,for,for,ADP,IN,xxx,True,True
8,$,$,SYM,$,$,False,False
9,6,6,NUM,CD,d,False,False


Con esta utilidad podemos inspeccionar varias propiedades del documento procesado:

|Tag|Descrición|doc2[0].tag|
|:------|:------:|:------|
|`.text`|The original word text<!-- .element: style="text-align:left;" -->|`Tesla`|
|`.lemma_`|The base form of the word|`tesla`|
|`.pos_`|The simple part-of-speech tag|`PROPN`/`proper noun`|
|`.tag_`|The detailed part-of-speech tag|`NNP`/`noun, proper singular`|
|`.shape_`|The word shape – capitalization, punctuation, digits|`Xxxxx`|
|`.is_alpha`|Is the token an alpha character?|`True`|
|`.is_stop`|Is the token part of a stop list, i.e. the most common words of the language?|`False`|

## Objetos Span
Un span puede interpretarse como una porción de un documento, es decir, puede empezar desde alún índice hasta otro. Esto facilita el procesamiento por pedazos (chunks) en lugar el documento completo.

In [11]:
# Definition of NLP according to Wikipedia 
doc = nlp(u"Natural language processing (NLP) is a subfield of computer science, \
information engineering, and artificial intelligence concerned with the \
interactions between computers and human (natural) languages, in particular \
how to program computers to process and analyze large amounts of natural language data.\
Challenges in natural language processing frequently involve speech recognition, natural \
language understanding, and natural language generation.")

quote = doc[10:30]
quote

computer science, information engineering, and artificial intelligence concerned with the interactions between computers and human (natural)

Observemos aquí que el slice es por los tokens y no por los caracteres individuales. Esto es muy útil ya que podemos estar seguros de no interrumpir abruptamente los tokens.

## Trabajando con oraciones
Podemos iterar sobre oraciones en los documentos, es decir, frases separadas por el punto "."

In [12]:
doc = nlp("This is the first sentence. This is the second sentence. And this is the last sentence.")
for sent in doc.sents:
    print(sent)

This is the first sentence.
This is the second sentence.
And this is the last sentence.


**Nota:** Cada punto es considerado un token, entonces en el segundo "This" en el anterior documento está en el índice `6`, no en el `5`.

In [13]:
print(f"Token 5: {doc[5]}")
print(f"Token 6: {doc[6]}")
print(f"Is token 6 a sentence start? {doc[6].is_sent_start}")

Token 5: .
Token 6: This
Is token 6 a sentence start? True


## Tokenización, entidades y visualización

Con la base anterior ya podemos estudiar más de cerca cómo spaCy tokeniza cadenas ambiguas, qué hace con correos o URLs y cómo expone entidades y trozos de sustantivos para construir reglas o sistemas híbridos.

In [14]:
import spacy
nlp = spacy.load("en_core_web_sm")

mystring = '"We\'re moving to L.A.!"'
print(mystring)

"We're moving to L.A.!"


## Tokenización con spaCy

Al inicio hemos dicho que los tokens son unos "elementos" que usamos para dividir el corpus. He usado la palabra elementos porque un token no está restringido únicamente a palabras, un token pueden ser n-gramas o incluso cada letra puede ser considerado un token. Todo depende de la tarea y el nivel de especificidad.

Por defecto, spaCy tokeniza por palabras y algunos signos de puntuación. Observemos lo que hace spaCy con la oración que hemos definido antes.

In [15]:
doc = nlp(mystring)
for token in doc:
    print(token)

"
We
're
moving
to
L.A.
!
"


spaCy es capaz de entender textos complejos como correos electrónicos y direcciones web, además del rol que cada caracter juega en la oración.

In [16]:
doc2 = nlp(u"We're here to help! Send snail-mail, email support@oursite.com or visit us at http://www.oursite.com!")

for t in doc2:
    print(t)

We
're
here
to
help
!
Send
snail
-
mail
,
email
support@oursite.com
or
visit
us
at
http://www.oursite.com
!


Observemos que en este caso el caracter punto (.) es utilizado en el correo y la url pero no fue interpretado como un token independiente.

## Entidades
spaCy tiene el poder de inferir entidades o sustantivos.

In [17]:
doc = nlp("Apple to build a Hong Kong factory for $6 million.")
for entity in doc.ents:
    print(entity)
    print(entity.label_)
    print(str(spacy.explain(entity.label_)))
    print('\n')

Apple
ORG
Companies, agencies, institutions, etc.


Hong Kong
GPE
Countries, cities, states


$6 million
MONEY
Monetary values, including unit




Observemos como la libreria es capaz de pre-clasificar algunas palabras en cuanto a que posiblemente hacen referencia.

##  Trozos de sustantivos
spaCy es capaz de detectar sustantivos compuestos, es decir sustantivos que están compuestos por más de una palabra.

In [18]:
doc = nlp("Autonomous cars shift insurance liability toward manufacturers.")
for chunk in doc.noun_chunks:
    print(chunk)

Autonomous cars
insurance liability
manufacturers


Observemos que en el ejemplo anterior "Autonomous car" es un sustantivo compuesto y spacy es capaz de identificarlo como un trozo.

Este es un caso particular en el inglés, en los diccionarios en español podrían haber otras peculiaridades.

## displaCy

Este es un modulo para visualizar objetos spacy. Resulta muy útil para dibujar la relación semantica detectada entre los tokens.

In [19]:
from spacy import displacy

doc = nlp("Apple is going to build a U.K. factory for $6 million.")
# dep for syntactic dependency
displacy.render(doc, style='dep', jupyter=True, options={'distance': 110})

In [20]:
doc = nlp("Over the last quarter Apple sold nearly 20 thousand iPds for  profit of $6 million.")
# dep for syntactic dependency
displacy.render(doc, style='ent', jupyter=True)